# VeloceReduction — Reduce one observing night

This notebook is the master workflow for reducing a complete Veloce observing
night.

**Input:** `observations/YYMMDD/`  
**Output:** `reductions/vr_X.Y.Z/YYMMDD/`

Detailed algorithms are implemented in the `velocereduction` Python modules.
This notebook intentionally contains only the main reduction steps.

## 0. Setup

Choose the night and reduction settings.

When run interactively as a notebook, the values below are used directly.
When converted to `reduce_night.py`, the night is supplied on the command line.

In [ ]:
import sys
import argparse

import numpy as np

from astropy.table import Table

import matplotlib.pyplot as plt

from velocereduction import (
    __version__,
    utils,
    flat,
    tramlines,
    calibration,
    wavelength
)


def running_in_notebook():
    """True for an interactive Jupyter notebook, False for reduce_night.py."""
    if 'ipykernel' in sys.modules:
        return True
    else:
        return False


IN_NOTEBOOK = running_in_notebook()

In [ ]:
if IN_NOTEBOOK:

    # -------------------------------------------------------------------------
    # Interactive notebook settings
    # -------------------------------------------------------------------------

    night = '001122'            # Observing night in YYMMDD format,
                                # e.g. 001122, which is the reference night
    # night = '260703'            

    log_level = 'DEBUG'          # DEBUG / INFO / WARNING / ERROR
    diagnostics = 'full'       # none / basic / full

    extraction_mode = 'summed'  # summed / fibre
    overwrite = False

else:

    # -------------------------------------------------------------------------
    # Command-line settings for reduce_night.py
    # -------------------------------------------------------------------------

    parser = argparse.ArgumentParser(
        description='Reduce one complete Veloce observing night.'
    )

    parser.add_argument(
        'night',
        help='Observing night in YYMMDD format, e.g. 001122'
    )

    parser.add_argument(
        '--log-level',
        choices=['DEBUG', 'INFO', 'WARNING', 'ERROR'],
        default='INFO'
    )

    parser.add_argument(
        '--diagnostics',
        choices=['none', 'basic', 'full'],
        default='basic'
    )

    parser.add_argument(
        '--extraction-mode',
        choices=['summed', 'fibre'],
        default='summed'
    )

    parser.add_argument(
        '--overwrite',
        action='store_true'
    )

    args = parser.parse_args()

    night = args.night
    log_level = args.log_level
    diagnostics = args.diagnostics
    extraction_mode = args.extraction_mode
    overwrite = args.overwrite

In [ ]:
config = utils.ReductionConfig(
    night=night,
    log_level=log_level,
    diagnostics=diagnostics,
    extraction_mode=extraction_mode,
    overwrite=overwrite,
)

paths = utils.prepare_reduction(config, version=__version__)
logger = utils.setup_logging(config, paths)

logger.info('Starting VeloceReduction %s for night %s', __version__, night)
logger.info('Diagnostics: %s', diagnostics)
logger.info('Extraction mode: %s', extraction_mode)

print(f'\nNight:             {night}')
print(f'VeloceReduction:   {__version__}')
print(f'Logging:           {log_level}')
print(f'Diagnostics:       {diagnostics}')
print(f'Extraction:        {extraction_mode}')
print(f'Output:            {paths.root}')

## 1. Identify observations

**Input:** raw observations and observing log  
**Output:** night overview and `reduction_input_YYMMDD.txt`

All observations are classified once at the beginning of the reduction.
Later stages use this table rather than repeatedly reading and classifying
raw FITS headers.

In [ ]:
reduction_input = utils.identify_observations(config, paths)

utils.write_reduction_input(reduction_input, config, paths)

if IN_NOTEBOOK:
    display(reduction_input)

## 2. Measure displacement relative to reference night

Measure the displacement of each detector relative to the reference night.

SimTh images are used for all three CCDs, with SimLC providing an additional
registration measurement for CCD2 and CCD3.

In [ ]:
detector_shifts = tramlines.measure_detector_shifts(
    reduction_input,
    config,
    paths,
)

if IN_NOTEBOOK:
    display(detector_shifts)

## 3. Master Flat, nightly tramlines, and Flat response

**Input:** Flat observations + reference tramlines + detector shifts  
**Output:** master Flat, nightly tramline geometry, extracted Flat, response Flat, and blaze function

Create a high-S/N master Flat in detector coordinates and use it to determine
the nightly tramline geometry. The Flat is then extracted along the fitted
tramlines and smoothed along the fibre direction to separate the small-scale
detector response from the smooth illumination and blaze.

In [ ]:
master_flat = flat.create_master_flat(
    reduction_input,
    config,
    paths,
)

In [ ]:
nightly_tramlines = tramlines.fit_nightly_tramlines(
    reduction_input,
    master_flat,
    detector_shifts,
    config,
    paths,
)

if (
    IN_NOTEBOOK
    and config.diagnostics != 'none'
):
    tramlines.show_summary(
        nightly_tramlines
    )

In [ ]:
flat_products = flat.create_flat_products(
    master_flat,
    nightly_tramlines,
    config,
    paths,
)

## 6. Extract wavelength-calibration observations

**Input:** SimLC and FibTh observations  
**Output:** time-stamped extracted calibration spectra

SimLC and FibTh are both used to establish the wavelength solution.

In [ ]:
calibration_spectra = tramlines.extract_calibration_spectra(
    reduction_input,
    nightly_tramlines,
    config,
    paths
)

In [ ]:
for calibration_type in ['SimTh','SimLC','FibTh']:
    tramlines.save_calibration_spectra(
        calibration_spectra[calibration_type],
        paths.wavelength / f'{calibration_type.lower()}.fits',
        calibration_type=calibration_type,
        overwrite=True,
    )

In [ ]:
calibration_peak_tables = calibration.measure_calibration_peaks_for_night(
    calibration_spectra,
    output_dir=paths.wavelength,
    diagnostic_dir=paths.figures / 'calibration',
    maximum_signal={
        'SimLC': None,
        'SimTh': None,
        'FibTh': None,
    },
    log_level=config.log_level,
    diagnostics=config.diagnostics,
    overwrite=config.overwrite
)

## 7. Fit the wavelength solution

**Input:** extracted SimLC + FibTh spectra and their MJD timestamps  
**Output:** wavelength calibration model for the night

In [ ]:
initial_wavelength_solution = (
    wavelength.load_initial_wavelength_solutions(
        paths.repository
    )
)

calibration_peak_tables["SimLC"] = (
    wavelength.identify_simlc_peaks(
        calibration_peak_tables["SimLC"],
        initial_wavelength_solution["SimLC"],
        detector_shifts,
    )
)

In [ ]:
simlc_wavelength_solutions = {}

for ccd in [2, 3]:

    simlc_wavelength_solutions[ccd] = (
        wavelength.fit_and_save_wavelength_surface(
            calibration_peak_tables["SimLC"],

            calibration_type="SimLC",
            ccd=ccd,

            degree_y=7,
            degree_m=5,

            minimum_y=200,
            maximum_y=3900,
            minimum_signal_to_noise=20,
            maximum_y_uncertainty=0.01,

            output_dir=paths.wavelength,
            diagnostic_dir=(
                paths.figures
                / "wavelength"
            ),

            diagnostics=config.diagnostics,
            overwrite=True,
        )
    )

# AND NOW ON TO FibTh and SimTh!

In [ ]:
import sys
sys.exit()

In [ ]:
import pickle

# ThAr atlas from Murphy et al. (2007)
dtype = [
    ("wavenumber", float),
    ("wave_air", float),
    ("log10_intensity", float),
    ("element", "U10"),
    ("ion", "U10"),
    ("source", "U2"),
]
thar_lines = Table(np.genfromtxt(
    paths.repository / 'velocereduction/veloce_reference_data/thar_UVES_MM090311.dat',
    dtype=dtype,
    comments="#",
    autostrip=True
))
thar_lines['wave_vac'] = utils.wavelength_air_to_vac(thar_lines['wave_air'])

delta_wavelength = 0.05
thar_lines_fine_murphy = dict()
thar_lines_fine_murphy['wave_vac'] = np.arange(thar_lines['wave_vac'][0], thar_lines['wave_vac'][-1]+delta_wavelength, delta_wavelength)
thar_lines_fine_murphy['wave_air'] = utils.wavelength_vac_to_air(thar_lines_fine_murphy['wave_vac'])
thar_lines_fine_murphy['intensity'] = np.zeros(len(thar_lines_fine_murphy['wave_vac']))
for index in range(len(thar_lines_fine_murphy['wave_vac'])):
    thar_line_in_range = np.where((thar_lines['wave_vac']-0.5*delta_wavelength < thar_lines_fine_murphy['wave_vac'][index]) & (thar_lines_fine_murphy['wave_vac'][index] < thar_lines['wave_vac']+0.5*delta_wavelength))[0]
    if len(thar_line_in_range) > 0:
        thar_lines_fine_murphy['intensity'][index] = np.mean(10**thar_lines['log10_intensity'][thar_line_in_range])

In [ ]:
nist_th_file = paths.repository / 'velocereduction/veloce_reference_data/th_linelist_NIST.pickle'
with open(nist_th_file, 'rb') as f:
    atomic_data_dict = pickle.load(f)
th_lines = dict()
th_lines['wave_air']  = atomic_data_dict['linelist']['obs_wl_air(nm)']*10
th_lines['wave_vac']  = utils.wavelength_air_to_vac(atomic_data_dict['linelist']['obs_wl_air(nm)']*10)
th_lines['intensity'] = atomic_data_dict['linelist']['intens']

delta_wavelength = 0.05
th_lines_fine_nist = dict()
th_lines_fine_nist['wave_vac'] = np.arange(th_lines['wave_vac'][0], th_lines['wave_vac'][-1]+delta_wavelength, delta_wavelength)
th_lines_fine_nist['wave_air'] = utils.wavelength_vac_to_air(th_lines_fine_nist['wave_vac'])
th_lines_fine_nist['intensity'] = np.zeros(len(th_lines_fine_nist['wave_vac']))
for index in range(len(th_lines_fine_nist['wave_vac'])):
    th_line_in_range = np.where((th_lines['wave_vac']-0.5*delta_wavelength < th_lines_fine_nist['wave_vac'][index]) & (th_lines_fine_nist['wave_vac'][index] < th_lines['wave_vac']+0.5*delta_wavelength))[0]
    if len(th_line_in_range) > 0:
        th_lines_fine_nist['intensity'][index] = np.mean(th_lines['intensity'][th_line_in_range])

In [ ]:
def plot_th(order, ccd, y0=2048):
    # def calc_(y0):

    try:
        coefficients = np.loadtxt(paths.repository / f'velocereduction/wavelength_coefficients/wavelength_coefficients_ccd_{ccd}_order_{order}_lc.txt')
    except:
        coefficients = np.loadtxt(paths.repository / f'velocereduction/wavelength_coefficients/wavelength_coefficients_ccd_{ccd}_order_{order}_thxe.txt')
        
    simth_counts = calibration_spectra['SimTh'][str(ccd)][0]['counts'][104-order]

    y = np.arange(4112)
    # y0 = np.median(y)   # 2055.5

    detector_shift_y = detector_shifts['dy'][
        detector_shifts['ccd'] == int(ccd)
    ][0]

    y_centered = y - y0

    wavelength_reference = np.polynomial.polynomial.polyval(
        y_centered,
        coefficients,
    )*10

    wavelength_min = wavelength_reference.min()
    wavelength_max = wavelength_reference.max()

    th_lines_fine_nist_in_range = (
        (th_lines_fine_nist['wave_vac'] > wavelength_min)
        & (th_lines_fine_nist['wave_vac'] < wavelength_max)
    )

    th_lines_fine_murphy_in_range = (
        (thar_lines_fine_murphy['wave_vac'] > wavelength_min)
        & (thar_lines_fine_murphy['wave_vac'] < wavelength_max)
    )
    th_lines_fine_scidoc_in_range = (
        (th_lines_fine_scidoc['wave_vac'] > wavelength_min)
        & (th_lines_fine_scidoc['wave_vac'] < wavelength_max)
    )
    
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(15, 8),
        sharex=True,
    )

    axes[0].plot(
        wavelength_reference,
        # np.log10(simth_counts),
        simth_counts.clip(max=500),
        lw=0.8,
    )

    axes[0].plot(
        thar_lines_fine_murphy['wave_vac'][th_lines_fine_murphy_in_range],
        - 500 * thar_lines_fine_murphy['intensity'][th_lines_fine_murphy_in_range] / np.max(thar_lines_fine_murphy['intensity'][th_lines_fine_murphy_in_range]),
        color='C1',
        lw=0.5,
    )
    axes[0].plot(
        th_lines_fine_nist['wave_vac'][th_lines_fine_nist_in_range],
        - 500 * th_lines_fine_nist['intensity'][th_lines_fine_nist_in_range] / np.max(th_lines_fine_nist['intensity'][th_lines_fine_nist_in_range]),
        color='C3',
        lw=0.5,
    )
    axes[0].plot(
        th_lines_fine_scidoc['wave_vac'][th_lines_fine_scidoc_in_range],
        - 100 * th_lines_fine_scidoc['intensity'][th_lines_fine_scidoc_in_range] / np.max(th_lines_fine_scidoc['intensity'][th_lines_fine_scidoc_in_range]),
        color='C4',
        lw=0.5,
    )

    # for th_lines_fine_nist_vacuum_i, th_lines_fine_nist_air_i, th_lines_fine_nist_intensity_i in zip(
    #     th_lines_fine_nist['wave_vac'][th_lines_fine_nist_in_range],
    #     th_lines_fine_nist['wave_air'][th_lines_fine_nist_in_range],
    #     th_lines_fine_nist['intensity'][th_lines_fine_nist_in_range],
    # ):
        # axes[0].text(
        #     th_lines_fine_nist_vacuum_i,
        #     th_lines_fine_nist_intensity_i,
        #     f'{th_lines_fine_nist_air_i:.4f}',
        #     rotation=90,
        #     fontsize=6,
        #     ha='center',
        #     va='bottom',
        #     c = 'C0'
        # )
        # axes[0].scatter(
        #     th_lines_fine_nist_vacuum_i,
        #     th_lines_fine_nist_intensity_i,
        #     s=10,
        #     c='C0',
        #     alpha=0.5,
        # )
        # axes[0].axvline(
        #     th_lines_fine_nist_vacuum_i,
        #     lw=0.7,
        #     alpha=0.5,
        #     c = 'C0'
        # )

    # for murphy_wavelength_vacuum_i, murphy_wavelength_air_i, murphy_intensity_i in zip(
    #     thar_lines_fine_murphy['wave_vac'][th_lines_fine_murphy_in_range],
    #     thar_lines_fine_murphy['wave_air'][th_lines_fine_murphy_in_range],
    #     thar_lines_fine_murphy['intensity'][th_lines_fine_murphy_in_range],
    # ):
        # axes[0].text(
        #     murphy_wavelength_vacuum_i,
        #     murphy_intensity_i,
        #     f'{murphy_wavelength_air_i:.4f}',
        #     rotation=90,
        #     fontsize=6,
        #     ha='center',
        #     va='bottom',
        #     c = 'C1'
        # )
        # axes[0].scatter(
        #     murphy_wavelength_vacuum_i,
        #     murphy_intensity_i,
        #     s=10,
        #     c='C1',
        #     alpha=0.5,
        # )
        # axes[0].axvline(
        #     murphy_wavelength_vacuum_i,
        #     lw=0.7,
        #     alpha=0.5,
        #     c = 'C1'
        # )


    axes[0].set_ylabel('Wavelength')
    axes[0].set_title(
        'Murphy Th wavelengths: AIR'
    )


    axes[1].plot(
        wavelength_reference,
        np.log10(simth_counts.clip(min=1)),
        lw=0.8,
    )

    axes[1].set_ylabel('Counts')
    axes[1].set_xlabel(
        'Dispersion pixel $y$'
    )

    axes[1].set_title(
        'Murphy Th wavelengths: VACUUM'
    )

    plt.show()
    plt.close()

plot_th(order=103, ccd=3, y0=2048)

## 8. Dark Products

In [ ]:
# dark_products = utils.process_darks(
#     processed,
#     config,
#     paths
# ) 

## 9. Extract science spectra

**Input:** processed Science images + tramlines + Flat products + wavelength model  
**Output:** wavelength-calibrated Science spectra

The current default extraction is summed across the Science fibres.
A fibre-resolved extraction can use the same interface in future.

In [ ]:
# science_spectra = extraction.extract_science(
#     processed.science,
#     nightly_tramlines,
#     flat_products,
#     wavelength_solution,
#     config,
#     paths
# )

## 10. Velocities

**Input:** wavelength-calibrated Science spectra  
**Output:** barycentric corrections and first-pass radial velocities

In [ ]:
# science_spectra = velocities.add_barycentric_corrections(
#     science_spectra,
#     config,
#     paths
# )

# science_spectra = velocities.measure_initial_rvs(
#     science_spectra,
#     template='solar',
#     config=config,
#     paths=paths
# )

## 11. B-star and telluric products

**Input:** B-star observations  
**Output:** extracted B-star spectra and telluric measurements

In [ ]:
# bstar_spectra = extraction.extract_science(
#     processed.bstars,
#     nightly_tramlines,
#     flat_products,
#     wavelength_solution,
#     config,
#     paths,
#     product_type='Bstar'
# )

# telluric_products = tellurics.measure_tellurics(
#     bstar_spectra,
#     config,
#     paths
# )

## 12. Final products and reduction summary

Write the final Science FITS products, retained diagnostic figures, and
human-readable summary of the night.

In [ ]:
# utils.write_science_products(
#     science_spectra,
#     config,
#     paths
# )

# summary = utils.create_reduction_summary(
#     reduction_input=reduction_input,
#     detector_shifts=detector_shifts,
#     tramlines=nightly_tramlines,
#     wavelength_solution=wavelength_solution,
#     science_spectra=science_spectra,
#     config=config,
#     paths=paths
# )

# utils.write_reduction_summary(summary, config, paths)

In [ ]:
logger.info('Reduction completed successfully.')

print()
print('Reduction complete')
print(f'Night:    {night}')
print(f'Version:  {__version__}')
print(f'Products: {paths.root}')
print(f'Summary:  {paths.reduction_summary}')